In [1]:
!pip install matplotlib
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- Dimensionless Constants ---
# Distance: 1 AU (Astronomical Unit)
# Time: 1 Earth Year
# Mass: 1 Solar Mass
G = 4 * np.pi**2 

def compute_accelerations(positions, masses):
    """
    Computes the gravitational acceleration for all bodies in 3D space.
    positions: numpy array of shape (N, 3)
    masses: numpy array of shape (N,)
    """
    N = len(masses)
    accelerations = np.zeros((N, 3))
    
    for i in range(N):
        for j in range(N):
            if i != j:
                r_vec = positions[j] - positions[i]
                r_mag = np.linalg.norm(r_vec)
                # Apply Newton's Law of Universal Gravitation
                accelerations[i] += G * masses[j] * r_vec / (r_mag**3)
                
    return accelerations

def rk4_step(positions, velocities, masses, dt):
    """
    Advances the system by one time step using 4th-Order Runge-Kutta.
    """
    k1_v = compute_accelerations(positions, masses)
    k1_r = velocities
    
    k2_v = compute_accelerations(positions + 0.5 * dt * k1_r, masses)
    k2_r = velocities + 0.5 * dt * k1_v
    
    k3_v = compute_accelerations(positions + 0.5 * dt * k2_r, masses)
    k3_r = velocities + 0.5 * dt * k2_v
    
    k4_v = compute_accelerations(positions + dt * k3_r, masses)
    k4_r = velocities + dt * k3_v
    
    new_positions = positions + (dt / 6.0) * (k1_r + 2*k2_r + 2*k3_r + k4_r)
    new_velocities = velocities + (dt / 6.0) * (k1_v + 2*k2_v + 2*k3_v + k4_v)
    
    return new_positions, new_velocities

# --- Initialization ---
# Masses: [Sun, Earth, Mars, Spaceship] (Spaceship mass is negligible)
masses = np.array([1.0, 3.003e-6, 3.213e-7, 0.0])

# Initial Positions (x, y, z) in AU
# Assuming a simplified starting alignment on the x-axis for Earth and Mars
positions = np.array([
    [0.0, 0.0, 0.0],       # Sun at origin
    [1.0, 0.0, 0.0],       # Earth at 1 AU
    [1.524, 0.0, 0.0],     # Mars at 1.524 AU
    [1.0, 0.0, 0.0]        # Spaceship starting at Earth
])

# Initial Velocities (vx, vy, vz) in AU/Year
# Circular orbit velocity is roughly v = sqrt(G * M_sun / r)
v_earth = np.sqrt(G * masses[0] / 1.0)
v_mars = np.sqrt(G * masses[0] / 1.524)

# Spaceship requires Earth's velocity + a departure delta-v for Hohmann transfer
# To be optimized later in the project
v_spacecraft = v_earth + 0.6  

velocities = np.array([
    [0.0, 0.0, 0.0],          # Sun (stationary in this frame)
    [0.0, v_earth, 0.0],      # Earth moving along y-axis
    [0.0, v_mars, 0.0],       # Mars moving along y-axis
    [0.0, v_spacecraft, 0.0]  # Spaceship departure velocity
])

# Simulation parameters
dt = 0.001  # Time step in years
total_time = 1.0  # Simulate for 1 Earth year
num_steps = int(total_time / dt)

# Arrays to store trajectory history for 3D visualization
history_pos = np.zeros((num_steps, len(masses), 3))

# Main simulation loop
for step in range(num_steps):
    history_pos[step] = positions
    positions, velocities = rk4_step(positions, velocities, masses, dt)
# --- Additional Numerical Solvers ---

def euler_step(positions, velocities, masses, dt):
    """
    Advances the system using the basic Euler method.
    Expected to show high error and orbital drift over time.
    """
    accelerations = compute_accelerations(positions, masses)
    
    new_positions = positions + velocities * dt
    new_velocities = velocities + accelerations * dt
    
    return new_positions, new_velocities

def leapfrog_step(positions, velocities, accelerations, masses, dt):
    """
    Advances the system using the Leapfrog (Velocity Verlet) method.
    Conserves energy better than RK4 over very long simulations.
    Requires passing the current accelerations to avoid redundant calculations.
    """
    # Half-step velocity update
    v_half = velocities + 0.5 * accelerations * dt
    
    # Full-step position update
    new_positions = positions + v_half * dt
    
    # Update accelerations at new positions
    new_accelerations = compute_accelerations(new_positions, masses)
    
    # Full-step velocity update
    new_velocities = v_half + 0.5 * new_accelerations * dt
    
    return new_positions, new_velocities, new_accelerations


# --- 3D Visualization ---

def plot_3d_trajectory(history_pos, title="Interplanetary Trajectory"):
    """
    Plots the 3D paths of all bodies in the simulation.
    history_pos: numpy array of shape (num_steps, num_bodies, 3)
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    labels = ['Sun', 'Earth', 'Mars', 'Spaceship']
    colors = ['orange', 'blue', 'red', 'green']
    marker_sizes = [300, 100, 80, 50]
    
    # Plot trajectories and final positions
    for i in range(len(labels)):
        # Extract X, Y, Z coordinates for body i across all time steps
        x_coords = history_pos[:, i, 0]
        y_coords = history_pos[:, i, 1]
        z_coords = history_pos[:, i, 2]
        
        # Plot the orbital path
        ax.plot(x_coords, y_coords, z_coords, label=f"{labels[i]} Path", color=colors[i], alpha=0.7)
        
        # Plot the final position as a solid marker
        ax.scatter(x_coords[-1], y_coords[-1], z_coords[-1], 
                   color=colors[i], s=marker_sizes[i], label=f"{labels[i]} Final", edgecolors='black')

    # Format the plot
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('X Position (AU)')
    ax.set_ylabel('Y Position (AU)')
    ax.set_zlabel('Z Position (AU)')
    
    # Keep the aspect ratio equal so orbits don't look distorted
    ax.set_box_aspect([1, 1, 1]) 
    
    # Adjust axes limits based on Mars' orbit (~1.5 AU)
    ax.set_xlim([-1.6, 1.6])
    ax.set_ylim([-1.6, 1.6])
    ax.set_zlim([-1.6, 1.6])
    
    # Place legend outside the plot area
    ax.legend(loc='center left', bbox_to_anchor=(1.1, 0.5))
    plt.tight_layout()
    plt.show()

# --- Example of How to Run and Plot the RK4 Data ---
# Assuming 'history_pos' was populated by the RK4 loop from the previous step:
# plot_3d_trajectory(history_pos, title="Earth to Mars Transfer (RK4 Method)")


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


/tmp/ipykernel_16794/1461389607.py:27: RuntimeWarning: invalid value encountered in divide
  accelerations[i] += G * masses[j] * r_vec / (r_mag**3)
